# SupplyPrescript
## Notebook 7: Integrated Prediction & Prescriptive Decision Pipeline

### Objective
Notebook 7 connects the two major analytical components developed previously:

1. Predictive Analytics — estimate shipment-delay probability using the tuned XGBoost model.
2. Prescriptive Analytics — use the predicted risk as input to the PuLP optimization engine and recommend an operational action.

### End-to-End Flow
Shipment Input → Tuned XGBoost → Delay Probability → Risk Level → Optimization → Recommended Action



## 7.1 Import Required Libraries


In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from pulp import (
    LpProblem,
    LpMinimize,
    LpVariable,
    LpStatus,
    lpSum,
    value,
    PULP_CBC_CMD
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Libraries imported successfully.")


Libraries imported successfully.


## 7.2 Define Project Paths


In [2]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(BASE_DIR, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")

print("Project directory:", BASE_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Models directory:", MODELS_DIR)


Project directory: c:\Users\USER\OneDrive\Desktop\VSCode\c++\SupplyPrescript
Processed data directory: c:\Users\USER\OneDrive\Desktop\VSCode\c++\SupplyPrescript\data\processed
Models directory: c:\Users\USER\OneDrive\Desktop\VSCode\c++\SupplyPrescript\models


## 7.3 Load the Tuned XGBoost Model


In [3]:
MODEL_PATH = os.path.join(MODELS_DIR, "Tuned_XGBoost.pkl")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Tuned model not found at: {MODEL_PATH}. "
        "Run Notebook 6 first and save Tuned_XGBoost.pkl."
    )

tuned_model = joblib.load(MODEL_PATH)

print("Tuned XGBoost model loaded successfully.")
print(type(tuned_model))


Tuned XGBoost model loaded successfully.
<class 'xgboost.sklearn.XGBClassifier'>


## 7.4 Load Prepared Test Data


In [4]:
X_test_path = os.path.join(PROCESSED_DIR, "X_test.csv")
y_test_path = os.path.join(PROCESSED_DIR, "y_test.csv")

if not os.path.exists(X_test_path):
    raise FileNotFoundError(f"Missing file: {X_test_path}")

if not os.path.exists(y_test_path):
    raise FileNotFoundError(f"Missing file: {y_test_path}")

X_test = pd.read_csv(X_test_path)
y_test = pd.read_csv(y_test_path).squeeze()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


X_test shape: (36104, 33)
y_test shape: (36104,)


## 7.5 Verify Feature Compatibility


In [5]:
model_features = list(tuned_model.get_booster().feature_names)

print("Number of model features:", len(model_features))
print("Number of test features :", X_test.shape[1])

missing_features = [
    feature for feature in model_features
    if feature not in X_test.columns
]

extra_features = [
    feature for feature in X_test.columns
    if feature not in model_features
]

print("Missing model features:", missing_features)
print("Extra test features:", extra_features)


Number of model features: 33
Number of test features : 33
Missing model features: []
Extra test features: []


In [6]:
if missing_features:
    raise ValueError(
        "X_test is missing features required by the tuned model."
    )

X_test_model = X_test[model_features].copy()

print("Feature compatibility check passed.")
print("Final input shape:", X_test_model.shape)


Feature compatibility check passed.
Final input shape: (36104, 33)


## 7.6 Predict Shipment Delay Probability


In [7]:
delay_probabilities = tuned_model.predict_proba(
    X_test_model
)[:, 1]

predicted_classes = (
    delay_probabilities >= 0.50
).astype(int)

print("First 10 delay probabilities:")
print(delay_probabilities[:10])

print("\nFirst 10 predicted classes:")
print(predicted_classes[:10])


First 10 delay probabilities:
[2.4511601e-04 3.6734642e-04 5.9267182e-05 2.3460498e-01 7.0521206e-01
 5.5393712e-03 9.9993515e-01 3.5608691e-04 9.9975401e-01 9.9988854e-01]

First 10 predicted classes:
[0 0 0 0 1 0 1 0 1 1]


## 7.7 Convert Probability into Risk Level

Prototype thresholds:
- Low Risk: probability < 0.40
- Medium Risk: 0.40 ≤ probability < 0.70
- High Risk: probability ≥ 0.70

These thresholds are business-policy assumptions and can be calibrated later.


In [8]:
def classify_risk(delay_probability):
    if delay_probability >= 0.70:
        return "High Risk"
    elif delay_probability >= 0.40:
        return "Medium Risk"
    else:
        return "Low Risk"

risk_levels = [
    classify_risk(probability)
    for probability in delay_probabilities
]

print(pd.Series(risk_levels).value_counts())


High Risk      19583
Low Risk       15330
Medium Risk     1191
Name: count, dtype: int64


## 7.8 Create Prediction Results Table


In [9]:
prediction_results = pd.DataFrame({
    "Delay_Probability": delay_probabilities,
    "Predicted_Delay": predicted_classes,
    "Risk_Level": risk_levels
})

prediction_results.head(10)


,Delay_Probability,Predicted_Delay,Risk_Level
0,0.000245,0,Low Risk
1,0.000367,0,Low Risk
2,0.000059,0,Low Risk
3,0.234605,0,Low Risk
4,0.705212,1,High Risk
5,0.005539,0,Low Risk
6,0.999935,1,High Risk
7,0.000356,0,Low Risk
8,0.999754,1,High Risk
9,0.999889,1,High Risk


## 7.9 Evaluate the Integrated Prediction Step


In [10]:
prediction_accuracy = accuracy_score(y_test, predicted_classes)
prediction_precision = precision_score(
    y_test, predicted_classes, zero_division=0
)
prediction_recall = recall_score(
    y_test, predicted_classes, zero_division=0
)
prediction_f1 = f1_score(
    y_test, predicted_classes, zero_division=0
)
prediction_auc = roc_auc_score(
    y_test, delay_probabilities
)

print("Integrated Prediction Performance")
print("----------------------------------")
print("Accuracy :", prediction_accuracy)
print("Precision:", prediction_precision)
print("Recall   :", prediction_recall)
print("F1 Score :", prediction_f1)
print("ROC-AUC  :", prediction_auc)


Integrated Prediction Performance
----------------------------------
Accuracy : 0.9685907378683802
Precision: 0.9567750146857255
Recall   : 0.9873206708425945
F1 Score : 0.9718078758949881
ROC-AUC  : 0.9961252757647255


## 7.10 Define the Prescriptive Optimization Engine


In [11]:
def optimize_shipment(
    delay_probability,
    budget=20000,
    delay_penalty_weight=10000
):
    """Recommend an operational action based on predicted delay probability.

    Action costs and delay impacts are prototype assumptions and should
    be replaced with validated business data before production use.
    """

    action_costs = {
        "standard_shipping": 0,
        "air_freight": 15000,
        "alternative_supplier": 10000,
        "delay_launch": 5000
    }

    delay_impact = {
        "standard_shipping": delay_probability,
        "air_freight": 0.20,
        "alternative_supplier": 0.30,
        "delay_launch": 1.00
    }

    decision_vars = {
        action: LpVariable(action, cat="Binary")
        for action in action_costs
    }

    problem = LpProblem(
        "SupplyPrescript_Optimization",
        LpMinimize
    )

    problem += lpSum(
        decision_vars[action] * (
            action_costs[action]
            + delay_impact[action] * delay_penalty_weight
        )
        for action in action_costs
    )

    problem += lpSum(decision_vars.values()) == 1

    problem += lpSum(
        decision_vars[action] * action_costs[action]
        for action in action_costs
    ) <= budget

    problem.solve(PULP_CBC_CMD(msg=False))

    status = LpStatus[problem.status]

    if status != "Optimal":
        return {
            "status": status,
            "recommended_action": None,
            "additional_cost": None,
            "expected_delay_risk": None,
            "objective_value": None
        }

    recommended_action = None

    for action, variable in decision_vars.items():
        if variable.value() == 1:
            recommended_action = action
            break

    return {
        "status": status,
        "recommended_action": recommended_action,
        "additional_cost": action_costs[recommended_action],
        "expected_delay_risk": delay_impact[recommended_action],
        "objective_value": value(problem.objective)
    }


## 7.11 Create the Complete SupplyPrescript Decision Function

This is the central component of Notebook 7. It takes one prepared shipment feature vector, predicts its delay probability, classifies its risk, and sends the prediction to the optimization engine.


In [12]:
def supplyprescript_decision(
    shipment_features,
    budget=20000
):
    """Run the complete prediction-to-recommendation pipeline."""

    if not isinstance(shipment_features, pd.DataFrame):
        raise TypeError(
            "shipment_features must be a pandas DataFrame."
        )

    if len(shipment_features) != 1:
        raise ValueError(
            "shipment_features must contain exactly one shipment."
        )

    missing = [
        feature for feature in model_features
        if feature not in shipment_features.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required model features: {missing}"
        )

    shipment_for_model = shipment_features[
        model_features
    ].copy()

    probability = float(
        tuned_model.predict_proba(
            shipment_for_model
        )[0, 1]
    )

    predicted_delay = int(probability >= 0.50)
    risk_level = classify_risk(probability)

    optimization = optimize_shipment(
        delay_probability=probability,
        budget=budget
    )

    return {
        "delay_probability": probability,
        "predicted_delay": predicted_delay,
        "risk_level": risk_level,
        "recommended_action": optimization["recommended_action"],
        "additional_cost": optimization["additional_cost"],
        "expected_delay_risk": optimization["expected_delay_risk"],
        "optimization_status": optimization["status"]
    }


## 7.12 Test the Complete Pipeline


In [13]:
sample_shipment = X_test_model.iloc[[0]].copy()

sample_result = supplyprescript_decision(
    sample_shipment,
    budget=20000
)

sample_result


{'delay_probability': 0.000245116010773927,
 'predicted_delay': 0,
 'risk_level': 'Low Risk',
 'recommended_action': 'standard_shipping',
 'additional_cost': 0,
 'expected_delay_risk': 0.000245116010773927,
 'optimization_status': 'Optimal'}

## 7.13 Display the Decision in a Human-Readable Format


In [14]:
print("======================================")
print("       SUPPLYPRESCRIPT DECISION")
print("======================================")
print(f"Delay Probability : {sample_result['delay_probability']:.2%}")
print(f"Predicted Delay   : {sample_result['predicted_delay']}")
print(f"Risk Level        : {sample_result['risk_level']}")
print(f"Recommended Action: {sample_result['recommended_action']}")
print(f"Additional Cost   : ${sample_result['additional_cost']:,.2f}")
print(
    f"Expected Delay Risk: "
    f"{sample_result['expected_delay_risk']:.2%}"
)
print(f"Optimization Status: {sample_result['optimization_status']}")
print("======================================")


       SUPPLYPRESCRIPT DECISION
Delay Probability : 0.02%
Predicted Delay   : 0
Risk Level        : Low Risk
Recommended Action: standard_shipping
Additional Cost   : $0.00
Expected Delay Risk: 0.02%
Optimization Status: Optimal


## 7.14 Run the Pipeline for Multiple Shipments


In [15]:
sample_size = min(25, len(X_test_model))
batch_results = []

for index in range(sample_size):
    shipment = X_test_model.iloc[[index]]

    result = supplyprescript_decision(
        shipment,
        budget=20000
    )

    result["shipment_index"] = index
    batch_results.append(result)

batch_results_df = pd.DataFrame(batch_results)

batch_results_df.head(10)


,delay_probability,predicted_delay,risk_level,recommended_action,additional_cost,expected_delay_risk,optimization_status,shipment_index
0,0.000245,0,Low Risk,standard_shipping,0,0.000245,Optimal,0
1,0.000367,0,Low Risk,standard_shipping,0,0.000367,Optimal,1
2,0.000059,0,Low Risk,standard_shipping,0,0.000059,Optimal,2
3,0.234605,0,Low Risk,standard_shipping,0,0.234605,Optimal,3
4,0.705212,1,High Risk,standard_shipping,0,0.705212,Optimal,4
5,0.005539,0,Low Risk,standard_shipping,0,0.005539,Optimal,5
6,0.999935,1,High Risk,standard_shipping,0,0.999935,Optimal,6
7,0.000356,0,Low Risk,standard_shipping,0,0.000356,Optimal,7
8,0.999754,1,High Risk,standard_shipping,0,0.999754,Optimal,8
9,0.999889,1,High Risk,standard_shipping,0,0.999889,Optimal,9


## 7.15 Analyze Recommended Actions


In [16]:
print("Recommended Action Distribution:")
print(
    batch_results_df["recommended_action"].value_counts()
)

print("\nRisk Level Distribution:")
print(
    batch_results_df["risk_level"].value_counts()
)


Recommended Action Distribution:
recommended_action
standard_shipping    25
Name: count, dtype: int64

Risk Level Distribution:
risk_level
High Risk    14
Low Risk     11
Name: count, dtype: int64


## 7.16 Create the Final Decision Table


In [17]:
final_decision_table = batch_results_df[
    [
        "shipment_index",
        "delay_probability",
        "predicted_delay",
        "risk_level",
        "recommended_action",
        "additional_cost",
        "expected_delay_risk",
        "optimization_status"
    ]
].copy()

final_decision_table


,shipment_index,delay_probability,predicted_delay,risk_level,recommended_action,additional_cost,expected_delay_risk,optimization_status
0,0,0.000245,0,Low Risk,standard_shipping,0,0.000245,Optimal
1,1,0.000367,0,Low Risk,standard_shipping,0,0.000367,Optimal
2,2,0.000059,0,Low Risk,standard_shipping,0,0.000059,Optimal
3,3,0.234605,0,Low Risk,standard_shipping,0,0.234605,Optimal
4,4,0.705212,1,High Risk,standard_shipping,0,0.705212,Optimal
5,5,0.005539,0,Low Risk,standard_shipping,0,0.005539,Optimal
6,6,0.999935,1,High Risk,standard_shipping,0,0.999935,Optimal
7,7,0.000356,0,Low Risk,standard_shipping,0,0.000356,Optimal
8,8,0.999754,1,High Risk,standard_shipping,0,0.999754,Optimal
9,9,0.999889,1,High Risk,standard_shipping,0,0.999889,Optimal


## 7.17 Save Integrated Decision Results


In [18]:
OUTPUT_PATH = os.path.join(
    PROCESSED_DIR,
    "SupplyPrescript_decision_results.csv"
)

os.makedirs(PROCESSED_DIR, exist_ok=True)

final_decision_table.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Decision results saved to:")
print(OUTPUT_PATH)


Decision results saved to:
c:\Users\USER\OneDrive\Desktop\VSCode\c++\SupplyPrescript\data\processed\SupplyPrescript_decision_results.csv


## 7.18 Save the Model Feature List


In [19]:
FEATURES_PATH = os.path.join(
    MODELS_DIR,
    "model_features.csv"
)

os.makedirs(MODELS_DIR, exist_ok=True)

pd.DataFrame({
    "Feature": model_features,
    "Position": range(len(model_features))
}).to_csv(
    FEATURES_PATH,
    index=False
)

print("Model feature list saved to:")
print(FEATURES_PATH)


Model feature list saved to:
c:\Users\USER\OneDrive\Desktop\VSCode\c++\SupplyPrescript\models\model_features.csv


## 7.19 Validate the Saved Decision Output


In [20]:
saved_results = pd.read_csv(OUTPUT_PATH)

print("Saved results shape:", saved_results.shape)
print("Saved result columns:")
print(saved_results.columns.tolist())

saved_results.head()


Saved results shape: (25, 8)
Saved result columns:
['shipment_index', 'delay_probability', 'predicted_delay', 'risk_level', 'recommended_action', 'additional_cost', 'expected_delay_risk', 'optimization_status']


,shipment_index,delay_probability,predicted_delay,risk_level,recommended_action,additional_cost,expected_delay_risk,optimization_status
0,0,0.000245,0,Low Risk,standard_shipping,0,0.000245,Optimal
1,1,0.000367,0,Low Risk,standard_shipping,0,0.000367,Optimal
2,2,0.000059,0,Low Risk,standard_shipping,0,0.000059,Optimal
3,3,0.234605,0,Low Risk,standard_shipping,0,0.234605,Optimal
4,4,0.705212,1,High Risk,standard_shipping,0,0.705212,Optimal


# Summary

Notebook 7 integrates the predictive and prescriptive components of SupplyPrescript.

### Predictive Layer
The tuned XGBoost model produces:
- Delay probability
- Predicted delay class
- Risk level

### Prescriptive Layer
The PuLP optimization engine uses the predicted delay probability to select an operational action while respecting the available budget.

### Integrated Output
Shipment → Delay Probability → Risk Level → Recommended Action → Cost / Expected Risk

### Files Generated
- `models/Tuned_XGBoost.pkl` (loaded from Notebook 6)
- `models/model_features.csv`
- `data/processed/SupplyPrescript_decision_results.csv`

### Important Project Limitation
The optimization costs and expected delay impacts are prototype assumptions. They should be replaced with validated business data before deployment.

### Next Stage
The next notebook can focus on closed-loop analytics and decision logging: recording the recommendation, manager decision, eventual outcome, and data needed for the system to learn from operator decisions.


# End of Notebook 7
